# 17 · Reentrenamiento progresivo de Ridge
Greta Solange Díaz Vega · Seminario de Tesis II · UNI

Cada enero se amplía el entrenamiento con objetivos anteriores al inicio del año. Se pronostican los meses siguientes con información hasta t−1 y los mismos horizontes de los cuadernos 14 y 15.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/gretadive/correlacion_actividad_solar_elnino/751cd7c187616d8b7ed7b175280dc80a4e999a29/data/processed/dataset_integrado_precipitacion.csv"
archivo = Path("datos.csv")
df = pd.read_csv(archivo if archivo.exists() else url, parse_dates=["fecha"])
df = df.set_index("fecha").sort_index().asfreq("MS")
variables = ["ICEN", "ONI", "precipitacion_mm"]
assert len(df) == 917 and df[variables + ["SSN"]].notna().all().all()
desfases = [0, 3, 6, 12, 24]

horizontes = [3,6,12,24]
numero = 17
print(df.index.min().date(), df.index.max().date(), len(df), "meses")

1950-01-01 2026-05-01 917 meses


## Análisis

In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline

# Penalizaciones seleccionadas en los cuadernos 14–15 con validación anterior a 2010.
alfas = {("ICEN",False):[0.1,0.1,0.1,100], ("ICEN",True):[0.1,0.1,100,100],
         ("ONI",False):[0.1,0.1,0.1,10], ("ONI",True):[0.1,0.1,0.1,100]}
filas = []
for indice in ["ICEN","ONI"]:
    for solar in [False,True]:
        variables = [indice,"SSN"] if solar else [indice]
        X = pd.concat([df[v].shift(k).rename(f"{v}_{k}") for v in variables for k in range(1,13)],axis=1)
        for h,alfa in zip(horizontes,alfas[indice,solar]):
            y = df[indice].shift(-h)
            destino = df.index + pd.DateOffset(months=h)
            valido = X.notna().all(axis=1) & y.notna()
            for anio in range(2010,2027):
                corte = pd.Timestamp(anio,1,1)
                prueba = valido & (df.index.year==anio)
                if not prueba.any():
                    continue
                ajuste = valido & (destino < corte)
                modelo = make_pipeline(StandardScaler(),Ridge(alpha=alfa))
                modelo.fit(X[ajuste],y[ajuste])
                for fecha,valor in zip(X[prueba].index,modelo.predict(X[prueba])):
                    filas.append([indice,h,solar,fecha,y.loc[fecha],valor])


## Evaluación en los mismos meses
MAE es el error absoluto medio. RMSE da más peso a los errores grandes. Compara las filas con y sin SSN del mismo índice y horizonte.

In [3]:
pred = pd.DataFrame(filas,columns=["indice","h","con_SSN","origen","observado","pronostico"])
pred["destino"] = [fecha+pd.DateOffset(months=int(h)) for fecha,h in zip(pred.origen,pred.h)]
assert not pred.duplicated(["indice","h","con_SSN","origen"]).any()
assert pred.groupby(["indice","h","origen"]).size().eq(2).all()
assert np.isfinite(pred.pronostico).all()
pred["error"] = pred.pronostico-pred.observado
metricas = pred.groupby(["indice","h","con_SSN"]).error.agg(
    N="size", MAE=lambda e:e.abs().mean(), RMSE=lambda e:np.sqrt((e**2).mean()))
pred.to_csv(f"{numero}_pronosticos.csv",index=False)
metricas.to_csv(f"{numero}_metricas.csv")
print(metricas.round(3).to_string())


                     N    MAE   RMSE
indice h  con_SSN                   
ICEN   3  False    194  0.547  0.706
          True     194  0.529  0.687
       6  False    191  0.769  0.957
          True     191  0.736  0.927
       12 False    185  0.843  1.025
          True     185  0.799  0.993
       24 False    173  0.872  1.045
          True     173  0.846  1.030
ONI    3  False    194  0.358  0.457
          True     194  0.358  0.453
       6  False    191  0.537  0.691
          True     191  0.530  0.680
       12 False    185  0.598  0.781
          True     185  0.576  0.772
       24 False    173  0.575  0.767
          True     173  0.571  0.771
